[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/predictive_maintenance.ipynb)

# 🔧 Predictive Maintenance — a beginner's guide

**Welcome!** This notebook assumes you know *almost nothing* about Machine Learning,
and explains every idea, word, and line of code as we go. By the end you'll have
trained real models that predict machine failure.

> This is one of two independent notebooks in this project. The other,
> `anomaly_detection.ipynb`, tackles a different problem. You can read them in any
> order — each stands on its own.

## 1 · The big picture — what are we even doing?

Imagine a factory full of machines. Machines wear out and **break**. A surprise
breakdown is expensive: production stops, parts get damaged, people wait.

**Predictive Maintenance (PdM)** means using data to fix a machine *just before* it
fails — not too early (wasteful) and not too late (breakdown). Think of replacing
your bike chain when it's worn but *before* it snaps mid-ride.

To do this we use **Machine Learning (ML)**: teaching a computer to spot patterns
from past examples, instead of writing fixed rules by hand. It's like showing a child
hundreds of photos of cats and dogs until they can tell a *new* animal apart — we
never wrote down "a cat has pointy ears", they *learned* it from examples.

We'll answer **two** predictive-maintenance questions, which need two kinds of ML:

| Question | Kind of ML | What the answer looks like |
|---|---|---|
| **Will this machine fail?** | **Classification** | a category: *fail* or *don't fail* |
| **How long until it fails?** | **Regression** | a number: *e.g. 42 cycles left* |

Both are **supervised learning** — meaning during practice we show the computer the
*correct answers* (called **labels**), like a teacher with an answer key.

## The tools (libraries) we'll use — in plain English

A **library** is a bundle of ready-made code someone else wrote so we don't have to.
Here are the ones this notebook uses:

| Library (short name) | What it does | Everyday analogy |
|---|---|---|
| **pandas** (`pd`) | Works with data in **tables** (rows & columns). A table is called a **DataFrame** (`df`). | A smart spreadsheet |
| **NumPy** (`np`) — *Numerical Python* | Fast maths on big lists of numbers (called **arrays**). | A super-fast calculator |
| **Matplotlib** (`plt`) & **Seaborn** (`sns`) | Draw charts and graphs. | Crayons for data |
| **scikit-learn** (`sklearn`) | A toolbox of ready-made ML models and helpers. | A box of LEGO machines |
| **PyTorch** (`torch`) | Builds **neural networks** (brain-inspired models). | LEGO for building a tiny brain |

> Whenever you see `pd.something` it means "use the pandas library to do something".

## 2 · Set up the environment

Run the cell below first. It prepares everything (and on Colab it downloads the
project and installs the libraries). You only need to run it once per session.

In [ ]:
# === Environment bootstrap — works on your laptop AND on Google Colab ========
# (Colab is a free website that runs Python notebooks in your browser, with a
#  free GPU. "GPU" = Graphics Processing Unit, a chip that makes ML training fast.)
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab the machine starts empty, so we download ("clone") the project
    # and install the libraries it needs.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Add the project folder to Python's search path so `from src import ...` works.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")
ROOT = _find_repo_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("Setup done. Running on", "Colab" if IN_COLAB else "your local machine.")

### Import the tools

"Importing" means telling Python which libraries we want to use in this notebook.
We give them short nicknames (`pd`, `np`, …) so we type less.

In [ ]:
import numpy as np                 # fast number crunching
import pandas as pd                # data tables (DataFrames)
import matplotlib.pyplot as plt    # charts
import seaborn as sns              # prettier charts + heatmaps
sns.set_theme(style="whitegrid")   # a clean default look for plots
pd.set_option("display.max_columns", 40)

# Our own project code (lives in the src/ folder):
from src import data, features, models, utils
print("All tools imported successfully ✅")

---
# Part A · "Will it fail?" — Classification

We start with the **AI4I 2020** dataset. (AI4I is just its name; it comes from the
**UCI** machine-learning repository — a famous free data library.)

## 3 · Meet the data

`data.load_ai4i()` downloads the dataset and hands us back a **DataFrame** (`df`) — a
table. Each of the 10,000 **rows** is one snapshot of a milling machine; each
**column** is something we measured about it.

In [ ]:
ai4i = data.load_ai4i()
print("The table has", ai4i.shape[0], "rows and", ai4i.shape[1], "columns.")
ai4i.head()   # .head() shows the first 5 rows

### What each column means (the "data dictionary")

| Column | Meaning |
|---|---|
| `Type` | product-quality grade — **L** (low, 50%), **M** (medium, 30%), **H** (high, 20%) |
| `Air temperature` | surrounding air temperature, in **K = Kelvin** (room ≈ 300 K). |
| `Process temperature` | the machine's own working temperature (K) |
| `Rotational speed` | how fast the tool spins, in **rpm = revolutions per minute**. |
| `Torque` | twisting force applied, in **Nm = Newton-metre**. |
| `Tool wear` | minutes the cutting tool has been used (wear builds up) |
| `Machine failure` | **the answer we want to predict** — 1 = failed, 0 = fine |
| `TWF, HDF, PWF, OSF, RNF` | five *specific* failure types. We'll **remove** these (explained later). |

Let's look at **one machine as a single record**, shown top-to-bottom so it's easy to
read (`.T` "transposes" — flips rows and columns):

In [ ]:
print("One example machine (row 0):")
display(ai4i.iloc[[0]].T)

### A quick numbers summary

`.describe()` gives basic statistics for each number column: the **mean** (average),
**std** (*standard deviation* — how spread out the values are), and the min/max.

In [ ]:
num_cols = ["Air temperature", "Process temperature",
            "Rotational speed", "Torque", "Tool wear"]
ai4i[num_cols].describe().T[["mean", "std", "min", "max"]].round(2)

## 4 · Look before you model — visualising the data

A golden rule in ML: **look at your data first.** Charts reveal patterns that raw
numbers hide. Let's make a few.

### 4a · How often do machines actually fail?

In [ ]:
counts = ai4i["Machine failure"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No failure (0)", "Failure (1)"], counts.values,
       color=["#4c9f70", "#d1495b"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f"{v}  ({100*v/len(ai4i):.1f}%)", ha="center")
ax.set_title("Most machines do NOT fail — the classes are imbalanced")
ax.set_ylabel("number of rows"); plt.show()

**What we learn:** only about **3.4%** of rows are failures. This is called **class
imbalance** — one outcome is far rarer than the other. It matters a lot: a lazy model
that *always* says "no failure" would be ~96.6% correct but utterly useless (it never
catches a real failure!). That's why later we won't trust *accuracy* alone.

### 4b · Do failed machines "look" different?

If failures had no pattern, prediction would be impossible. Let's overlay the
distribution of each measurement for failed vs healthy machines. A **distribution**
just shows which values are common (tall) vs rare (short).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), num_cols):
    for label, color in [(0, "#4c9f70"), (1, "#d1495b")]:
        subset = ai4i[ai4i["Machine failure"] == label][col]
        ax.hist(subset, bins=40, alpha=0.6, density=True,
                color=color, label="failed" if label else "healthy")
    ax.set_title(col); ax.legend()
axes.ravel()[-1].axis("off")  # hide the empty 6th panel
fig.suptitle("Measurement patterns: failed (red) vs healthy (green)", y=1.02)
fig.tight_layout(); plt.show()

Notice how failed machines tend to sit at the **extremes** — very high torque, high
tool wear, or unusual rotational speed. Those are the clues the model will learn.

### 4c · Which measurements move together? (correlation heatmap)

**Correlation** measures whether two things rise/fall together, on a scale from
**-1** (perfect opposites) through **0** (unrelated) to **+1** (perfectly in step). A
**heatmap** colours these so patterns pop out.

In [ ]:
corr = ai4i[num_cols + ["Machine failure"]].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation between measurements (and failure)")
plt.tight_layout(); plt.show()

For example, air and process temperature are strongly positive (they rise together),
which makes physical sense. Torque and rotational speed are negatively correlated.

### 4d · A closer look with box plots

A **box plot** summarises a distribution: the box covers the middle 50% of values,
the line inside is the **median** (middle value), and dots are unusual **outliers**.
Comparing healthy vs failed side by side highlights the gap.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["Torque", "Tool wear"]):
    sns.boxplot(data=ai4i, x="Machine failure", y=col, ax=ax,
                hue="Machine failure", legend=False, palette=["#4c9f70", "#d1495b"])
    ax.set_xticklabels(["healthy (0)", "failed (1)"])
    ax.set_title(col)
fig.tight_layout(); plt.show()

## 5 · Preparing the data for the model

Models can't read raw tables directly — we need to split things into:
- **X** (the "features"): the *inputs* the model learns from (the measurements).
- **y** (the "label"/"target"): the *answer* we want it to predict (failure: 0 or 1).

`features.prepare_ai4i()` does two important clean-ups:
1. **Removes "leakage" columns.** The five specific failure flags (TWF, HDF, …)
   basically *contain the answer*. Letting the model see them would be like giving a
   student the exam answers during practice — it would "cheat" and fail in real life.
2. **One-hot encodes** the `Type` column. Models need numbers, not words like "L/M/H".
   *One-hot encoding* turns one word-column into separate 0/1 columns (`type_l`,
   `type_m`, `type_h`).

In [ ]:
X, y = features.prepare_ai4i(ai4i)
print("Features (X) — the inputs the model sees:")
print(list(X.columns))
print("\nLabel (y) — what we predict:", y.name)
X.head()

### Train/test split — why we hide some data

We split the rows into two piles:
- **Training set** (75%): the model *learns* from these — its "homework".
- **Test set** (25%): kept hidden during training, used only to *grade* the model —
  its "final exam".

Why? Because anyone can memorise homework. The real question is whether the model
works on data it has **never seen**. `train_test_split` (from scikit-learn) does this.
We use `stratify=y` so the rare failures appear in the same proportion in both piles.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("Training rows:", len(X_train), " | Test rows:", len(X_test))

## 6 · Train the model — a Random Forest

We'll use a **Random Forest**. To understand it, start with a **decision tree**: a
flowchart of yes/no questions ("Is torque > 60? Is tool wear > 200?") that leads to a
guess. One tree alone is twitchy and over-confident.

A **Random Forest** grows *hundreds* of slightly different trees and lets them
**vote**. Like asking 300 doctors instead of 1 — the crowd's average is more reliable
than any single opinion. This "many-models-vote" idea is called an **ensemble**.

- `n_estimators=300` → grow 300 trees.
- `class_weight="balanced"` → pay extra attention to the rare failures (fixes the
  imbalance we saw).
- `.fit(...)` is the **training** step — "fit" = "learn from these examples".

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                             random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)   # <-- the model learns here
print("Trained a forest of", clf.n_estimators, "decision trees 🌳🌳🌳")

## 7 · How good is it? Grading the model

Now the exam. `clf.predict(...)` asks the model for its guesses on the **test set** it
never saw. We grade those guesses several ways — each tells us something different.

### 7a · Confusion matrix
A **confusion matrix** is a 2×2 table comparing *truth* vs *prediction*:

- **True Negative**: healthy, predicted healthy ✅
- **True Positive**: failed, predicted failed ✅
- **False Positive**: healthy, but we cried wolf ⚠️ (annoying)
- **False Negative**: failed, but we missed it ❌ (**dangerous** — surprise breakdown!)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
y_pred = clf.predict(X_test)              # the model's category guesses (0/1)
y_proba = clf.predict_proba(X_test)[:, 1] # its confidence (0..1) that it WILL fail

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["pred healthy", "pred fail"],
            yticklabels=["true healthy", "true fail"], ax=ax)
ax.set_title("Confusion matrix"); plt.show()

### 7b · Precision, Recall, F1 — the metrics that matter here

Because failures are rare, we focus on the **failure** row:

- **Recall** = of all the machines that *really* failed, what fraction did we catch?
  (Misses = dangerous, so we want recall high.)
- **Precision** = of all the machines we *flagged* as failing, what fraction truly did?
  (Low precision = lots of false alarms.)
- **F1-score** = a single balance of precision and recall (their "harmonic mean").

There's always a **trade-off**: catch more real failures (↑recall) and you usually
raise false alarms (↓precision). The right balance depends on costs.

In [ ]:
print(classification_report(y_test, y_pred, digits=3,
      target_names=["healthy (0)", "failure (1)"]))

### 7c · ROC curve and AUC

The model outputs a **probability** (0 to 1). We turn it into yes/no using a
**threshold** (default 0.5). Different thresholds give different precision/recall.

The **ROC curve** (Receiver Operating Characteristic — an old radar term, the name
doesn't matter) plots performance across *all* thresholds at once. The **AUC** (Area
Under the Curve) squashes that into one score:
- **1.0** = perfect, **0.5** = no better than a coin flip.

Intuitively, AUC = "if I pick one real failure and one healthy machine at random, how
often does the model rate the failure as more risky?"

In [ ]:
auc = roc_auc_score(y_test, y_proba)
fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax, name="Random Forest")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random guess")
ax.set_title(f"ROC curve — AUC = {auc:.3f}"); ax.legend(); plt.show()
print(f"ROC-AUC = {auc:.3f}  (closer to 1.0 is better)")

## 8 · Why did the model decide that? (explainability)

A model you can't explain is hard to trust on a factory floor. A Random Forest can
tell us **feature importance** — how much each input helped its decisions overall.

In [ ]:
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
imp.plot.barh(ax=ax, color="#3b7dd8")
ax.set_title("Which measurements matter most?"); plt.tight_layout(); plt.show()

### Going deeper: SHAP (optional)

Feature importance is *global* ("what matters in general"). **SHAP** explains a
*single* prediction — exactly how each measurement pushed *this* machine toward
"fail" or "healthy". (SHAP = SHapley Additive exPlanations, borrowed from game
theory.) It can be slow, so we sample a few hundred rows. If SHAP isn't installed the
cell simply skips — no problem.

In [ ]:
try:
    import shap
    sample = X_test.sample(min(300, len(X_test)), random_state=0)
    sv = shap.TreeExplainer(clf).shap_values(sample)
    sv = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv, sample, show=True)
except Exception as e:
    print("SHAP step skipped:", e)

---
# Part B · "How long until it fails?" — Regression

Predicting *yes/no* is useful, but knowing *how much life is left* is gold — you can
schedule the repair. This is **regression** (predicting a number), and we'll use a
different, richer dataset: **NASA's C-MAPSS** turbofan engines.

(C-MAPSS = *Commercial Modular Aero-Propulsion System Simulation* — NASA software that
simulated jet engines running until they wore out. We just use its output data.)

## 9 · Meet the engine data

This data is **time-series**: instead of one snapshot per machine, we have a *diary*
for each engine — one row per **cycle** (a unit of running time), from brand-new all
the way to failure. There are 100 engines.

In [ ]:
cm = data.load_cmapss("FD001")
train = cm["train"]
print("Training diary:", train.shape[0], "rows across",
      train["unit"].nunique(), "engines.")
train.head()

### What one row means

| Column | Meaning |
|---|---|
| `unit` | engine id (1–100). All rows with the same id = that engine's life story. |
| `cycle` | the time step: 1, 2, 3, … until the engine fails. |
| `op_setting_1..3` | the operating conditions that cycle (e.g. altitude, throttle). |
| `sensor_1..21` | 21 sensor readings (temperatures, pressures, spin speeds, …). |

One engine-cycle as a single record:

In [ ]:
print("Engine 1, cycle 1:")
display(train.iloc[[0]].T)

## 10 · Visualising wear — the heart of the idea

As an engine ages, its sensors **drift**. That drift is the signal we'll learn from.

### 10a · Sensors over one engine's lifetime

In [ ]:
e1 = train[train["unit"] == 1]
show = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, s in zip(axes.ravel(), show):
    ax.plot(e1["cycle"], e1[s], color="#3b7dd8")
    ax.set_title(s); ax.set_xlabel("cycle (age)")
fig.suptitle("Engine #1: sensors drift as it nears failure", y=1.02)
fig.tight_layout(); plt.show()

### 10b · Every engine fails at a different age

Each engine is plotted as one faint line (sensor_4 over its life). They all trend the
same way but end at **different lengths** — that's *why* we need a model to estimate
remaining life, instead of a fixed "replace at 200 cycles" rule.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for uid in train["unit"].unique():
    e = train[train["unit"] == uid]
    ax.plot(e["cycle"], e["sensor_4"], color="#3b7dd8", alpha=0.15)
ax.set_title("sensor_4 across all 100 engines (each line = one engine)")
ax.set_xlabel("cycle"); ax.set_ylabel("sensor_4"); plt.show()

lifetimes = train.groupby("unit")["cycle"].max()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(lifetimes, bins=25, color="#4c9f70")
ax.set_title(f"How long engines last (min {lifetimes.min()}, max {lifetimes.max()} cycles)")
ax.set_xlabel("lifetime in cycles"); plt.show()

## 11 · Creating the answer key (the RUL label)

The data doesn't come with the answer, so we **build** it. For each row we compute
**RUL** = *Remaining Useful Life* = (the engine's final cycle) − (this cycle). So a
brand-new engine has a high RUL; the last row before failure has RUL = 0.

**One clever twist (clipping):** when an engine is young and healthy, its sensors look
*identical* to a slightly-less-young healthy engine — there's no visible difference to
learn from. So predicting a huge RUL like "300" is impossible from the sensors. The
standard fix is to **cap** RUL at 125: we treat anything healthier than 125 as just
"plenty of life left (125)". The model then focuses on the period where wear actually
shows.

In [ ]:
train = features.add_rul(train, clip=125)
e1 = train[train["unit"] == 1]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(e1["cycle"], e1["rul"], color="#d1495b")
ax.set_title("Engine #1's RUL label: flat at 125 (healthy), then counts down to 0")
ax.set_xlabel("cycle"); ax.set_ylabel("RUL (cycles left)"); plt.show()

### Proof the sensors predict RUL

If sensors truly carry the signal, a sensor value should relate to RUL. Here's sensor_11
plotted against RUL for many engines — see the clear trend as RUL → 0.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sample = train.sample(3000, random_state=0)
ax.scatter(sample["rul"], sample["sensor_11"], s=6, alpha=0.3, color="#3b7dd8")
ax.set_xlabel("RUL (cycles left)"); ax.set_ylabel("sensor_11")
ax.set_title("sensor_11 shifts as the engine approaches failure (RUL → 0)")
ax.invert_xaxis(); plt.show()

## 12 · Feature engineering for time-series

**Feature engineering** = creating better inputs for the model. Two steps here:

1. **Drop dead sensors.** Some of the 21 sensors never change (flat lines) in this
   data — they're useless, so `features.feature_columns()` drops them.
2. **Add rolling features.** A single reading is a snapshot; failure is about *trends*.
   A **rolling mean** (average of the last few cycles) smooths noise and shows the
   trend; a **rolling standard deviation** shows if readings are getting jumpy. ("Rolling"
   = a window that slides along the timeline.)

In [ ]:
cols = features.feature_columns(train)
print("Useful sensors/settings kept:", len(cols))
train_fe = features.add_rolling_features(train, cols, window=5)
print("Columns after adding rolling features:", train_fe.shape[1])

### Reshaping into sequences (for the LSTM)

Our next model reads **sequences**, not single rows. So we slide a 30-cycle **window**
along each engine's diary; every window (shape: 30 cycles × N sensors) becomes one
training example, labelled with the RUL at the window's end. Think of it as showing
the model "the last 30 days" and asking "how much life is left now?"

In [ ]:
X_seq, y_seq = features.make_sequences(train_fe, cols, seq_len=30)
print("Sequence data shape:", X_seq.shape, " = (examples, timesteps, sensors)")

### Standardising (putting sensors on the same scale)

Sensors have wildly different ranges (one might be ~500, another ~0.03). Neural
networks learn best when inputs are on a **similar scale**. **Standardising** rescales
each sensor to have mean 0 and spread 1 — like converting everyone's test scores to a
common curve. We learn the scale from training data only, then apply it everywhere.

In [ ]:
scaler = utils.Standardizer().fit(X_seq.reshape(-1, X_seq.shape[-1]))
scale = lambda a: scaler.transform(a.reshape(-1, a.shape[-1])).reshape(a.shape)
X_seq_s = scale(X_seq)
print("Standardised. Example mean ≈", round(float(X_seq_s.mean()), 3),
      " spread ≈", round(float(X_seq_s.std()), 3))

## 13 · The model — an LSTM neural network

A **neural network** is a model loosely inspired by brain cells ("neurons") wired
together; it learns by adjusting the strength of those wires.

A plain network forgets the order of things. But an engine's *history order* matters,
so we use an **LSTM** = **Long Short-Term Memory** network. An LSTM reads the sequence
one step at a time and keeps a little **memory** of what it saw before — like reading
a story and remembering earlier chapters to understand the ending. (LSTM belongs to a
family called **RNN** = Recurrent Neural Networks.)

We train it to minimise **MSE** = *Mean Squared Error* = the average of the squared
differences between its guess and the truth. Squaring punishes big mistakes more. As
training proceeds, watch both the **training** loss and the **validation** loss (error
on a held-out slice) go down and stay close — a big gap would mean **overfitting**
(memorising instead of learning).

> On Colab's free GPU this is quick. On a plain laptop ~20 rounds ("epochs") takes a
> couple of minutes. An **epoch** = one full pass through the training data.

In [ ]:
model = models.LSTMRegressor(n_features=X_seq_s.shape[-1], hidden=64, layers=2)
history = models.train_lstm(model, X_seq_s, y_seq, epochs=20, batch_size=256)

utils.plot_loss(history, "LSTM learning curve (lower = better)"); plt.show()

## 14 · Test it on unseen engines

The **test** engines were stopped *before* failure; NASA tells us their true RUL. We
take each engine's most recent 30-cycle window and predict its RUL, then compare.

Two scores:
- **RMSE** = *Root Mean Squared Error* — average error **in cycles** (so "RMSE = 16"
  means "off by ~16 cycles on average"). Lower is better.
- **C-MAPSS score** — the official contest metric. It's **asymmetric**: predicting
  *too much* life left (optimistic → surprise failure) is punished harder than being
  *too cautious*. This mirrors real life, where a surprise breakdown costs more than
  an early check-up. Lower is better.

In [ ]:
X_test_seq = features.last_sequence_per_unit(cm["test"], cols, seq_len=30)
y_pred = models.predict_lstm(model, scale(X_test_seq))
y_true = cm["rul"]["rul"].to_numpy().clip(max=125)

print(f"RMSE         : {utils.rmse(y_true, y_pred):.1f} cycles  (avg error)")
print(f"C-MAPSS score: {utils.cmapss_score(y_true, y_pred):.0f}     (lower is better)")

### Reading the prediction chart

Each dot is one test engine: its **true** RUL (x-axis) vs our **predicted** RUL
(y-axis). Perfect predictions sit on the red diagonal line.
- Dots **below** the line = we predicted *less* life than real → **cautious/safe**.
- Dots **above** the line = we predicted *more* life than real → **risky** (these hurt
  the C-MAPSS score most).

In [ ]:
utils.plot_rul_scatter(y_true, y_pred); plt.show()

## 15 · Recap

You just built two predictive-maintenance models from scratch:

1. **Classification** (Random Forest) — *will* a machine fail? → high ROC-AUC.
2. **Regression** (LSTM) — *how soon*? → RMSE of ~15–20 cycles.

Along the way you met: features vs labels, train/test split, class imbalance,
confusion matrix, precision/recall, ROC-AUC, neural networks, LSTMs, standardising,
and feature engineering. That's a real ML toolkit! 🎉

**Next:** the sibling notebook `anomaly_detection.ipynb` tackles the *opposite*
situation — spotting trouble when you have **no labels at all**.

## 📖 Glossary — every abbreviation in one place

| Short | Full term | Meaning in one line |
|---|---|---|
| ML | Machine Learning | Teaching a computer to find patterns from examples instead of fixed rules. |
| IIoT | Industrial Internet of Things | Factory machines fitted with sensors that send data. |
| IoT | Internet of Things | Everyday devices connected to the internet. |
| PdM | Predictive Maintenance | Fixing a machine *just before* it breaks, using data. |
| RUL | Remaining Useful Life | How many cycles/hours a machine has left before failure. |
| LSTM | Long Short-Term Memory | A neural network that remembers earlier steps in a sequence. |
| RNN | Recurrent Neural Network | A network family that reads data step by step (LSTM is one). |
| MSE | Mean Squared Error | Average of the squared mistakes (how wrong, on average). |
| RMSE | Root MSE | Square root of MSE — error in the original units (e.g. cycles). |
| ROC | Receiver Operating Characteristic | A curve showing a classifier's trade-offs. |
| AUC | Area Under the Curve | One number (0–1) summarising the ROC curve; higher = better. |
| AE | Autoencoder | A network that learns to compress then rebuild data. |
| df | DataFrame | A table of data in pandas. |
| rpm | revolutions per minute | How fast something spins. |
| K | Kelvin | A temperature unit (0 K = absolute zero; room ≈ 300 K). |
| Nm | Newton-metre | A unit of torque (twisting force). |
| UCI | Univ. of California, Irvine | A famous free dataset repository. |
| NASA | (US space agency) | Source of the engine dataset. |
| C-MAPSS | Commercial Modular Aero-Propulsion System Simulation | NASA software that simulated the engine data. |